## Homework 00: Derivatives and Gradient Descent

Today you will refresh the calculus behind gradient-based learning.

Your main goal is to __derive and implement the gradients of MSE, MAE, L1, and L2 regularization terms__ in general __vector form__.

This homework is intentionally low-level: the goal is to see how losses turn into gradients before PyTorch computes them automatically.

We will work with a small preprocessed regression dataset stored in `boston_subset.json`.

In [ ]:
'''
Run this notebook from the `homework_00` folder, next to:
- `loss_and_derivatives.py`
- `boston_subset.json`

If you use Google Colab, upload the whole `homework_00` folder or upload these two files into the same runtime directory before running the notebook.
'''
print('Homework 00 files should be in the current directory.')

: 

In [2]:
# Run some setup code for this notebook.
import random
import numpy as np
import matplotlib.pyplot as plt

# If you edit loss_and_derivatives.py, restart the kernel or rerun the import cell below.

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
import json
with open('boston_subset.json', 'r') as iofile:
    dataset = json.load(iofile)
feature_matrix = np.array(dataset['data'])
targets = np.array(dataset['target'])

## Warming up: matrix differentiation
_These questions refresh the matrix calculus used by gradient descent and backpropagation._

The warm-up questions are adapted from older practical deep learning course materials.

For each inline question, include a short derivation or explanation. A final formula without reasoning is not enough for full credit.

Useful links: 
[1](http://www.atmos.washington.edu/~dennis/MatrixCalculus.pdf)
[2](http://cal.cs.illinois.edu/~johannes/research/matrix%20calculus.pdf)
[3](http://research.microsoft.com/en-us/um/people/cmbishop/prml/index.htm)

#### Inline question 1
$$  
y = x^Tx,  \quad x \in \mathbb{R}^N
$$

$$
\frac{dy}{dx} = 2x
$$

$$
y = x^T x = x_1^2 + x_2^2 + \cdots + x_n^2 = \sum_{i=1}^{n} x_i^2
$$

Taking the derivative with respect to \(x\),

$$
\frac{dy}{dx}
=
\frac{d}{dx}\left(\sum_{i=1}^{n} x_i^2\right)
=
\begin{bmatrix}
\frac{\partial}{\partial x_1}\sum_{i=1}^{n} x_i^2 \\
\frac{\partial}{\partial x_2}\sum_{i=1}^{n} x_i^2 \\
\vdots \\
\frac{\partial}{\partial x_n}\sum_{i=1}^{n} x_i^2
\end{bmatrix}
=
\begin{bmatrix}
2x_1 \\
2x_2 \\
\vdots \\
2x_n
\end{bmatrix}
=
2x
$$

#### Inline question 2
$$ 
y = tr(AB) \quad A,B \in \mathbb{R}^{N \times N} $$ 

where 
$$
tr(AB)=\sum_i (AB)_{ii}=\sum_i \sum_j A_{ij}B_{ji}
$$

$$
\frac{dy}{dA} = B^T
$$

$$
y = \operatorname{tr}(AB)
= \sum_{i=1}^{n} (AB)_{ii}
= \sum_{i=1}^{n} \sum_{k=1}^{n} A_{ik} B_{ki}
$$

Taking the derivative with respect to \(A\),

$$
\frac{dy}{dA}
=
\begin{bmatrix}
\frac{\partial y}{\partial A_{11}} & \frac{\partial y}{\partial A_{12}} & \cdots & \frac{\partial y}{\partial A_{1n}} \\
\frac{\partial y}{\partial A_{21}} & \frac{\partial y}{\partial A_{22}} & \cdots & \frac{\partial y}{\partial A_{2n}} \\
\vdots & \vdots & \ddots & \vdots \\
\frac{\partial y}{\partial A_{n1}} & \frac{\partial y}{\partial A_{n2}} & \cdots & \frac{\partial y}{\partial A_{nn}}
\end{bmatrix}
=
\begin{bmatrix}
B_{11} & B_{21} & \cdots & B_{n1} \\
B_{12} & B_{22} & \cdots & B_{n2} \\
\vdots & \vdots & \ddots & \vdots \\
B_{1n} & B_{2n} & \cdots & B_{nn}
\end{bmatrix}
=
B^T
$$

Note:

$$
\frac{\partial y}{\partial A_{ik}} = B_{ki}
$$

because all other terms are constants with respect to ${A_{ik}}$.

#### Inline question 3
$$  
y = x^TAc , \quad A\in \mathbb{R}^{N \times N}, x\in \mathbb{R}^{N}, c\in \mathbb{R}^{N} 
$$

$$
\frac{dy}{dx} = A c
$$

$$
\frac{dy}{dA} = x c ^ T
$$

Hint for the latter (one of the ways): use *ex. 2* result and the fact 
$$
tr(ABC) = tr (CAB)
$$

$$
(Ac)_i = \sum_{k=1}^{n} A_{ik} c_k
$$

$$
y = x^T A c
= \sum_{i=1}^{n} \sum_{k=1}^{n} x_i A_{ik} c_k
$$

Taking the derivative with respect to \(x\),

$$
\frac{dy}{dx}
=
\begin{bmatrix}
\frac{\partial y}{\partial x_1} \\
\frac{\partial y}{\partial x_2} \\
\vdots \\
\frac{\partial y}{\partial x_n}
\end{bmatrix}
=
\begin{bmatrix}
\sum_{k=1}^{n} A_{1k} c_k \\
\sum_{k=1}^{n} A_{2k} c_k \\
\vdots \\
\sum_{k=1}^{n} A_{nk} c_k
\end{bmatrix}
=
\begin{bmatrix}
(Ac)_1 \\
(Ac)_2 \\
\vdots \\
(Ac)_n
\end{bmatrix}
=
Ac
$$

## Loss functions and derivatives implementation
You will need to implement the methods from `loss_and_derivatives.py` to go further.
__In this assignment we ignore the bias term__, so the linear model takes simple form of 
$$
\hat{\mathbf{y}} = XW
$$
where no extra column of 1s is added to the $X$ matrix.

Implement the loss functions, regularization terms, and their derivatives with respect to the weight matrix. 

__For this assignment, you can ignore the bias term. The dataset is preprocessed for this setting.__

After editing `loss_and_derivatives.py`, rerun the import cell below. If Python still seems to use an old version, restart the notebook kernel and run the cells again.

In [ ]:
# Reload the class definition after editing loss_and_derivatives.py
try:
    del LossAndDerivatives
except:
    pass

from loss_and_derivatives import LossAndDerivatives

Note that in this case we compute the __MSE__ and __MAE__ for vector __y__. In the reference implementation, we average the error along the __y__ dimensionality as well.

E.g. for residuals vector $[1., 1., 1., 1.]$ the averaged error value will be $\frac{1}{4}(1. + 1. + 1. + 1.)$ 

This matters for the multiplier in the loss-function derivatives. You can also refer to the `.mse` method implementation, which is already available in `loss_and_derivatives.py`. The `.mae` method is provided as an additional implementation example.

In [ ]:
w = np.array([1., 1.])
x_n, y_n = feature_matrix, targets

Here come several asserts to check yourself:

In [ ]:
w = np.array([1., 1.])
x_n, y_n = feature_matrix, targets

# Repeating data to make everything multi-dimensional
w = np.vstack([w[None, :] + 0.27, w[None, :] + 0.22, w[None, :] + 0.45, w[None, :] + 0.1]).T
y_n = np.hstack([y_n[:, None], 2*y_n[:, None], 3*y_n[:, None], 4*y_n[:, None]])

In [ ]:
reference_mse_derivative = np.array([
    [ 7.32890068, 12.88731311, 18.82128365, 23.97731238],
    [ 9.55674399, 17.05397661, 24.98807528, 32.01723714]
])
reference_l2_reg_derivative = np.array([
    [2.54, 2.44, 2.9 , 2.2 ],
    [2.54, 2.44, 2.9 , 2.2 ]
])

assert np.allclose(
    reference_mse_derivative,
    LossAndDerivatives.mse_derivative(x_n, y_n, w), rtol=1e-3
), 'Something wrong with MSE derivative'

assert np.allclose(
    reference_l2_reg_derivative,
    LossAndDerivatives.l2_reg_derivative(w), rtol=1e-3
), 'Something wrong with L2 reg derivative'

print(
    'MSE derivative:\n{} \n\nL2 reg derivative:\n{}'.format(
        LossAndDerivatives.mse_derivative(x_n, y_n, w),
        LossAndDerivatives.l2_reg_derivative(w))
)

In [ ]:
reference_mae_derivative = np.array([
    [0.19708867, 0.19621798, 0.19621798, 0.19572906],
    [0.25574138, 0.25524507, 0.25524507, 0.25406404]
])
reference_l1_reg_derivative = np.array([
    [1., 1., 1., 1.],
    [1., 1., 1., 1.]
])

assert np.allclose(
    reference_mae_derivative,
    LossAndDerivatives.mae_derivative(x_n, y_n, w), rtol=1e-3
), 'Something wrong with MAE derivative'

assert np.allclose(
    reference_l1_reg_derivative,
    LossAndDerivatives.l1_reg_derivative(w), rtol=1e-3
), 'Something wrong with L1 reg derivative'

print(
    'MAE derivative:\n{} \n\nL1 reg derivative:\n{}'.format(
        LossAndDerivatives.mae_derivative(x_n, y_n, w),
        LossAndDerivatives.l1_reg_derivative(w))
)

### Gradient descent on the real data
Here comes small loop with gradient descent algorithm. We compute the gradient over the whole dataset.

In [ ]:
def get_w_by_grad(X, Y, w_0, loss_mode='mse', reg_mode=None, lr=0.05, n_steps=100, reg_coeff=0.05):
    if loss_mode == 'mse':
        loss_function = LossAndDerivatives.mse
        loss_derivative = LossAndDerivatives.mse_derivative
    elif loss_mode == 'mae':
        loss_function = LossAndDerivatives.mae
        loss_derivative = LossAndDerivatives.mae_derivative
    else:
        raise ValueError('Unknown loss function. Available loss functions: `mse`, `mae`')
    
    if reg_mode is None:
        reg_function = LossAndDerivatives.no_reg
        reg_derivative = LossAndDerivatives.no_reg_derivative # lambda w: np.zeros_like(w)
    elif reg_mode == 'l2':
        reg_function = LossAndDerivatives.l2_reg
        reg_derivative = LossAndDerivatives.l2_reg_derivative
    elif reg_mode == 'l1':
        reg_function = LossAndDerivatives.l1_reg
        reg_derivative = LossAndDerivatives.l1_reg_derivative
    else:
        raise ValueError('Unknown regularization mode. Available modes: `l1`, `l2`, None')
    
    
    w = w_0.copy()

    for i in range(n_steps):
        empirical_risk = loss_function(X, Y, w) + reg_coeff * reg_function(w)
        gradient = loss_derivative(X, Y, w) + reg_coeff * reg_derivative(w)
        gradient_norm = np.linalg.norm(gradient)
        if gradient_norm > 5.:
            gradient = gradient / gradient_norm * 5.
        w -= lr * gradient
        
        if i % 25 == 0:
            print('Step={}, loss={},\ngradient values={}\n'.format(i, empirical_risk, gradient))
    return w


Let's check how it works.

In [ ]:
# Initial weight matrix
w = np.ones((2,1), dtype=float)
y_n = targets[:, None] 

In [ ]:
w_grad = get_w_by_grad(x_n, y_n, w, loss_mode='mse', reg_mode='l2', n_steps=250)

### Comparing with `sklearn`
Finally, let's compare our model with `sklearn` implementation.

In [ ]:
from sklearn.linear_model import Ridge

In [ ]:
lr = Ridge(alpha=0.05)
lr.fit(x_n, y_n)
print('sklearn linear regression implementation delivers MSE = {}'.format(np.mean((lr.predict(x_n) - y_n)**2)))

In [ ]:
plt.scatter(x_n[:, -1], y_n[:, -1])
plt.scatter(x_n[:, -1], x_n.dot(w_grad)[:, -1], color='orange', label='Handwritten linear regression', linewidth=5)
plt.scatter(x_n[:, -1], lr.predict(x_n), color='cyan', label='sklearn Ridge')
plt.legend()
plt.show()

While the solutions may look like a bit different, remember, that handwritten linear regression was unable to fit the bias term, it was equal to $0$ by default.

### Submit your work
Submit your completed `loss_and_derivatives.py` file and this notebook with your written derivations. The notebook should run from the `homework_00` folder without downloading extra files.